# Beam Geometry And Fields

This notebook models the present MOT optical geometry as collimated cooling and repump beams that share the same three retroreflected paths: two orthogonal horizontal axes and one vertical axis. The horizontal paths carry `45°` glass-cell angle-of-incidence metadata, the vertical beam enters from the top, and all 12 beams remain collimated through the cell with `12.7 mm` diameter.

In [ ]:
%matplotlib widget
from pathlib import Path
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from pmot import (
    build_mot_beams,
    default_simulation_config,
    plot_apparatus_geometry_3d,
    plot_intensity_cloud_3d_by_polarization,
    plot_intensity_lineout,
    plot_scalar_field_slice,
    plot_scalar_field_surface,
    project_paths,
    sample_filtered_intensity_along_line,
    sample_intensity_along_line,
    sample_intensity_cloud_by_polarization,
    sample_intensity_slice,
)


In [ ]:
PATHS = project_paths(PROJECT_ROOT)
CONFIG = default_simulation_config()
MOT_BEAMS = build_mot_beams(CONFIG)

geometry_table = pd.DataFrame([
    {
        'label': beam.label,
        'family': beam.family,
        'axis_name': beam.axis_name,
        'propagation_sense': beam.propagation_sense,
        'circular_polarization': beam.circular_polarization,
        'wavelength_nm': 1e9 * beam.wavelength_m,
        'laser_frequency_thz': beam.laser_frequency_hz / 1e12,
        'detuning_mhz': beam.detuning_hz / 1e6,
        'power_w': beam.power_w,
        'beam_diameter_mm': 2e3 * beam.beam_radius_m,
        'cell_aoi_deg': next(axis.cell_angle_of_incidence_deg for axis in CONFIG.axes if axis.name == beam.axis_name),
    }
    for beam in MOT_BEAMS
])
geometry_table

In [ ]:
geometry_table[['label', 'family', 'axis_name', 'propagation_sense', 'circular_polarization', 'beam_diameter_mm', 'cell_aoi_deg']]

In [ ]:
AXIS_SCAN_DEFINITIONS = {
    'horizontal_x': ((-25e-3, 0.0, 0.0), (25e-3, 0.0, 0.0), 'x'),
    'horizontal_y': ((0.0, -25e-3, 0.0), (0.0, 25e-3, 0.0), 'y'),
    'vertical_z': ((0.0, 0.0, -25e-3), (0.0, 0.0, 25e-3), 'z'),
}

axis_profiles = {}
for axis_name, (start_m, stop_m, coordinate_label) in AXIS_SCAN_DEFINITIONS.items():
    axis_mm, axis_total = sample_filtered_intensity_along_line(
        MOT_BEAMS,
        start_m,
        stop_m,
        axis_name=axis_name,
    )
    _, right_handed = sample_filtered_intensity_along_line(
        MOT_BEAMS,
        start_m,
        stop_m,
        axis_name=axis_name,
        circular_polarization='right',
    )
    _, left_handed = sample_filtered_intensity_along_line(
        MOT_BEAMS,
        start_m,
        stop_m,
        axis_name=axis_name,
        circular_polarization='left',
    )
    _, cooling_right = sample_filtered_intensity_along_line(
        MOT_BEAMS,
        start_m,
        stop_m,
        axis_name=axis_name,
        family='cooling',
        circular_polarization='right',
    )
    _, cooling_left = sample_filtered_intensity_along_line(
        MOT_BEAMS,
        start_m,
        stop_m,
        axis_name=axis_name,
        family='cooling',
        circular_polarization='left',
    )
    _, repump_right = sample_filtered_intensity_along_line(
        MOT_BEAMS,
        start_m,
        stop_m,
        axis_name=axis_name,
        family='repump',
        circular_polarization='right',
    )
    _, repump_left = sample_filtered_intensity_along_line(
        MOT_BEAMS,
        start_m,
        stop_m,
        axis_name=axis_name,
        family='repump',
        circular_polarization='left',
    )
    axis_profiles[axis_name] = {
        'coordinate_label': coordinate_label,
        'distance_mm': axis_mm,
        'axis_total': axis_total,
        'right_handed': right_handed,
        'left_handed': left_handed,
        'cooling_right': cooling_right,
        'cooling_left': cooling_left,
        'repump_right': repump_right,
        'repump_left': repump_left,
    }

x_mm, total_x = sample_intensity_along_line(MOT_BEAMS, (-25e-3, 0.0, 0.0), (25e-3, 0.0, 0.0))
y_mm, total_y = sample_intensity_along_line(MOT_BEAMS, (0.0, -25e-3, 0.0), (0.0, 25e-3, 0.0))
z_mm, total_z = sample_intensity_along_line(MOT_BEAMS, (0.0, 0.0, -25e-3), (0.0, 0.0, 25e-3))


In [ ]:
plot_intensity_lineout(x_mm, total_x, 'Total MOT Intensity Along x', path=PATHS['outputs_fields'] / 'mot_total_lineout_x.png');
plot_intensity_lineout(y_mm, total_y, 'Total MOT Intensity Along y', path=PATHS['outputs_fields'] / 'mot_total_lineout_y.png');
plot_intensity_lineout(z_mm, total_z, 'Total MOT Intensity Along z', path=PATHS['outputs_fields'] / 'mot_total_lineout_z.png');

## Interactive Axis-Resolved Lineouts

The earlier lineout looked Gaussian because it mixed transverse Gaussian contributions from multiple beams and both optical families. The plots below isolate one apparatus direction at a time, so each panel only includes the beams that actually propagate along that axis. Right-handed traces correspond to the incident beams and left-handed traces correspond to the retroreflected beams.

In [ ]:
def draw_axis_resolved_lineouts(x_range_mm: float = 10.0):
    figure, axes = plt.subplots(3, 1, figsize=(10.5, 13.0), constrained_layout=True, sharex=False)
    figure.patch.set_facecolor('#fbfaf6')
    style_map = {
        'cooling_right': {'color': '#b91c1c', 'linestyle': '-', 'label': 'Cooling, right-handed'},
        'cooling_left': {'color': '#ef4444', 'linestyle': '--', 'label': 'Cooling, left-handed'},
        'repump_right': {'color': '#1d4ed8', 'linestyle': '-', 'label': 'Repump, right-handed'},
        'repump_left': {'color': '#60a5fa', 'linestyle': '--', 'label': 'Repump, left-handed'},
    }
    axis_order = [
        ('horizontal_x', 'Horizontal x beams'),
        ('horizontal_y', 'Horizontal y beams'),
        ('vertical_z', 'Vertical z beams'),
    ]
    for axis, (axis_name, title) in zip(axes, axis_order):
        axis.set_facecolor('#fbfaf6')
        profile = axis_profiles[axis_name]
        for key, style in style_map.items():
            axis.plot(profile['distance_mm'], profile[key], linewidth=2.2, **style)
        axis.plot(profile['distance_mm'], profile['axis_total'], color='#111827', linewidth=1.5, alpha=0.7, label='Axis total')
        axis.set_xlim(-x_range_mm, x_range_mm)
        axis.set_xlabel(f"{profile['coordinate_label']} coordinate [mm]")
        axis.set_ylabel('Intensity [W/m$^2$]')
        axis.set_title(title)
        axis.grid(True, alpha=0.28)
        axis.legend(loc='best', frameon=True, ncol=2)
    plt.show()

widgets.interact(
    draw_axis_resolved_lineouts,
    x_range_mm=widgets.FloatSlider(value=10.0, min=0.5, max=25.0, step=0.5, description='x-range [mm]')
);

In [ ]:
for plane in ('xy', 'xz', 'yz'):
    axis_1, axis_2, grid = sample_intensity_slice(
        MOT_BEAMS,
        plane=plane,
        extent_m=CONFIG.volume_extent_m,
        samples_per_axis=CONFIG.samples_per_axis,
    )
    plot_scalar_field_slice(
        axis_1,
        axis_2,
        grid,
        plane=plane,
        title=f'Total MOT Intensity Slice: {plane.upper()}',
        colorbar_label='Intensity [W/m$^2$]',
        path=PATHS['outputs_fields'] / f'mot_intensity_slice_{plane}.png',
        log_scale=True,
    );

## Interactive Surface Plots

In [ ]:
def draw_surface(plane: str = 'xy', extent_mm: float = 15.0, samples_per_axis: int = 121):
    axis_1, axis_2, grid = sample_intensity_slice(
        MOT_BEAMS,
        plane=plane,
        extent_m=extent_mm * 1e-3,
        samples_per_axis=samples_per_axis,
    )
    plot_scalar_field_surface(
        axis_1,
        axis_2,
        grid,
        plane=plane,
        title=f'Total MOT Intensity Surface: {plane.upper()}',
        z_label='Intensity [W/m$^2$]',
    )
    plt.show()

widgets.interact(
    draw_surface,
    plane=widgets.Dropdown(options=['xy', 'xz', 'yz'], value='xy', description='plane'),
    extent_mm=widgets.FloatSlider(value=15.0, min=2.0, max=50.0, step=1.0, description='extent [mm]'),
    samples_per_axis=widgets.IntSlider(value=121, min=41, max=201, step=20, description='samples')
);

In [ ]:
CLOUD_BY_POLARIZATION = sample_intensity_cloud_by_polarization(
    MOT_BEAMS,
    axial_extent_m=30e-3,
    axial_samples=17,
    radial_rings=3,
    angular_samples=12,
)
plot_intensity_cloud_3d_by_polarization(
    CLOUD_BY_POLARIZATION,
    title='3D Cooling And Repump Intensity Cloud By Polarization',
    path=PATHS['outputs_figures'] / 'mot_intensity_cloud_3d_by_polarization.png',
);
plot_apparatus_geometry_3d(MOT_BEAMS, path=PATHS['outputs_figures'] / 'mot_apparatus_geometry_3d.png');